In [ ]:
import os
os.environ["TRANSFORMERS_VERBOSITY"] = "error"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import joblib
import numpy as np
import pandas as pd
import re
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from transformers import pipeline

# ==========================
# PATHS
# ==========================

model_folder = "Models"

questions_path = os.path.join(model_folder, "questions.pkl")
answers_path = os.path.join(model_folder, "answers.pkl")
embeddings_path = os.path.join(model_folder, "embeddings.pkl")

excel_file = "Data/question_answers.xlsx"

# ==========================
# LOAD MODELS
# ==========================

print("Loading embedding model...")
model = SentenceTransformer("all-MiniLM-L6-v2")

print("Loading summarization model...")

summarizer = pipeline(
    "text-generation",
    model="google/flan-t5-base",
    tokenizer="google/flan-t5-base"
)

# ==========================
# LOAD TRAINED DATA
# ==========================

questions = joblib.load(questions_path)
answers = joblib.load(answers_path)
embeddings = joblib.load(embeddings_path)

# ==========================
# CLEAN TEXT FUNCTION
# ==========================

def clean_answer(text):

    text = text.replace('"', '')
    text = text.replace(',,', ',')
    text = text.replace('', '-')
    text = text.replace(',', ' ')

    text = re.sub(r'\s+', ' ', text)

    return text.strip()


# ==========================
# SUMMARIZATION FUNCTION
# ==========================

def summarize_answer(answer):

    if len(answer.split()) < 40:
        return answer

    prompt = f"AI Answer:-\n{answer}"

    result = summarizer(
        prompt,
        do_sample=False
    )

    return result[0]["generated_text"]


# ==========================
# CHATBOT LOOP
# ==========================

print("\nAI Chatbot Ready! Type 'exit' to stop.")

while True:

    user_question = input("\nAsk Question: ")
    print(f"User:-\n{user_question}")

    if user_question.lower() == "exit":
        print("Chatbot stopped.")
        break

    # Convert question to embedding
    user_embedding = model.encode([user_question])

    similarity = cosine_similarity(user_embedding, embeddings)

    best_index = similarity.argmax()
    score = similarity[0][best_index]

    # ==========================
    # MATCH FOUND
    # ==========================

    if score > 0.65:

        answer = answers[best_index]

        cleaned = clean_answer(answer)

        final_answer = summarize_answer(cleaned)
        print(final_answer)

    # ==========================
    # SELF LEARNING
    # ==========================

    else:

        print("\nI don't know the answer.")

        new_answer = input("Please provide the correct answer: ")

        questions.append(user_question)
        answers.append(new_answer)

        new_embedding = model.encode([user_question])

        embeddings = np.vstack([embeddings, new_embedding])

        # Save updated knowledge
        joblib.dump(questions, questions_path)
        joblib.dump(answers, answers_path)
        joblib.dump(embeddings, embeddings_path)

        # Update Excel
        df = pd.read_excel(excel_file)

        new_row = {
            "Category": "Learned",
            "Question": user_question,
            "Answer": new_answer,
            "number": len(df) + 1
        }

        df = pd.concat([df, pd.DataFrame([new_row])], ignore_index=True)

        df.to_excel(excel_file, index=False)

        print("\nNew knowledge learned and saved.")

c:\Users\ESR Solutions\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading embedding model...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2436.50it/s]


Loading summarization model...


Loading weights: 100%|██████████| 282/282 [00:00<00:00, 1419.54it/s]
The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'LlamaForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', 'DeepseekV3ForCausalLM', 'DiffLlamaForCausalLM', 'DogeForCausalLM', 'Dots1ForCausalLM', 'ElectraForCausalLM', 'Emu3ForCausalLM', 'ErnieForCausalLM', 'Ernie4_5ForCausalLM', 'Ernie4_5_MoeForCausalLM', 'Exaone4ForCausal


AI Chatbot Ready! Type 'exit' to stop.
User:-
Can i store car document in the app
Yes under each inventory profile youll find a Documents section. Upload and store files such as registration papers insurance pink slips or inspection certificates securely.
User:-
How to store car documwnt in the app
Yes under each inventory profile youll find a Documents section. Upload and store files such as registration papers insurance pink slips or inspection certificates securely.
